In [47]:
import pandas as pd
import numpy as np
import ast

from scipy.signal import resample
from scipy.stats import zscore
from tqdm import tqdm

In [48]:
CSV_PATH = "/content/mental_workload_dataset.csv"   # Change this

df = pd.read_csv("/Users/teljirisahith/Desktop/dataset/Single-Channel EEG Dataset for Task-Based Mental W/merged_eeg_datas_50_subjects.csv")

print(df.shape)
df.head()
print(df.columns)

(8249, 23)
Index(['task_id', 'student_id', 'neuro_delta', 'neuro_theta', 'neuro_loAlpha',
       'neuro_hiAlpha', 'neuro_loBeta', 'neuro_hiBeta', 'neuro_loGamma',
       'neuro_midGamma', 'neuro_attention', 'neuro_meditation',
       'neuro_poor_signal_strength', 'neuro_raw_eeg', 'neuro_relative_delta',
       'neuro_relative_theta', 'neuro_relative_alpha', 'neuro_relative_beta',
       'neuro_relative_gamma', 'neuro_dominan_rhythm', 'workload_score',
       'is_correct', 'created_at'],
      dtype='object')


In [49]:
signals = []
labels = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    
    signal = row["neuro_raw_eeg"]

    # Convert string to Python list
    if isinstance(signal, str):
        signal = ast.literal_eval(signal)

    signal = np.array(signal, dtype=np.float32)

    signals.append(signal)
    labels.append(row["workload_score"])

print("Total Signals :", len(signals))
print("Total Labels  :", len(labels))

print("\nFirst Signal Length :", len(signals[0]))
print("First Label :", labels[0])

100%|██████████| 8249/8249 [00:04<00:00, 1662.82it/s]

Total Signals : 8249
Total Labels  : 8249

First Signal Length : 1024
First Label : 0.0


In [50]:
from collections import Counter

lengths = [len(s) for s in signals]

print("Minimum Length :", min(lengths))
print("Maximum Length :", max(lengths))

print("\nTop 20 Most Common Lengths:\n")

for length, count in Counter(lengths).most_common(20):
    print(f"{length} samples : {count} recordings")

Minimum Length : 468
Maximum Length : 1536

Top 20 Most Common Lengths:

512 samples : 5481 recordings
1536 samples : 432 recordings
511 samples : 297 recordings
1024 samples : 276 recordings
510 samples : 173 recordings
509 samples : 144 recordings
506 samples : 144 recordings
508 samples : 136 recordings
507 samples : 135 recordings
505 samples : 96 recordings
504 samples : 90 recordings
502 samples : 83 recordings
503 samples : 79 recordings
500 samples : 70 recordings
501 samples : 58 recordings
499 samples : 45 recordings
498 samples : 39 recordings
1022 samples : 36 recordings
496 samples : 32 recordings
497 samples : 31 recordings


In [51]:
from scipy.signal import resample

TARGET_FS = 250
ORIGINAL_FS = 512

X = []
y = []

for signal, label in zip(signals, labels):

    # Skip very short recordings
    if len(signal) < ORIGINAL_FS:
        continue

    # Resample
    new_length = int(len(signal) * TARGET_FS / ORIGINAL_FS)
    signal_resampled = resample(signal, new_length)

    # Split into 1-second windows (250 samples)
    num_windows = len(signal_resampled) // TARGET_FS

    for i in range(num_windows):

        start = i * TARGET_FS
        end = start + TARGET_FS

        window = signal_resampled[start:end]

        if len(window) == TARGET_FS:
            X.append(window)
            y.append(label)

X = np.array(X, dtype=np.float32)
y = np.array(y)

print("X Shape :", X.shape)
print("y Shape :", y.shape)

X Shape : (7710, 250)
y Shape : (7710,)


In [52]:
from scipy.stats import zscore

X = np.array([zscore(window) for window in X], dtype=np.float32)

print("Mean of first window :", np.mean(X[0]))
print("Std of first window  :", np.std(X[0]))

Mean of first window : 3.528595e-08
Std of first window  : 0.99999994


In [53]:
print("Minimum :", y.min())
print("Maximum :", y.max())

print("Number of unique values:", len(np.unique(y)))

print("First 20 unique values:")
print(np.unique(y)[:20])

Minimum : 0.0
Maximum : 1.0
Number of unique values: 3
First 20 unique values:
[0.  0.5 1. ]


In [54]:
from collections import Counter

print(Counter(y))

Counter({np.float64(1.0): 3065, np.float64(0.0): 2450, np.float64(0.5): 2195})


In [55]:
# Keep only labels 0 and 1
mask = (y == 0.0) | (y == 1.0)

X = X[mask]
y = y[mask]

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nClass Distribution:")
unique, counts = np.unique(y, return_counts=True)

for label, count in zip(unique, counts):
    print(f"Class {int(label)} : {count}")

X shape: (5515, 250)
y shape: (5515,)

Class Distribution:
Class 0 : 2450
Class 1 : 3065


In [56]:
#write code to save this as nparrays 

In [57]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training Shape :", X_train.shape)
print("Testing Shape  :", X_test.shape)

print("\nTraining Labels")
print(np.unique(y_train, return_counts=True))

print("\nTesting Labels")
print(np.unique(y_test, return_counts=True))

Training Shape : (4412, 250)
Testing Shape  : (1103, 250)

Training Labels
(array([0., 1.]), array([1960, 2452]))

Testing Labels
(array([0., 1.]), array([490, 613]))


In [58]:
import torch

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

print(X_train.shape)
print(y_train.shape)

torch.Size([4412, 250])
torch.Size([4412])


In [59]:
X_train = X_train.unsqueeze(1)
X_test = X_test.unsqueeze(1)

print("Training :", X_train.shape)
print("Testing  :", X_test.shape)

Training : torch.Size([4412, 1, 250])
Testing  : torch.Size([1103, 1, 250])


In [60]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

print("Train batches :", len(train_loader))
print("Test batches  :", len(test_loader))

Train batches : 138
Test batches  : 35


In [61]:
import torch
import torch.nn as nn

class AttentionCNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(

            nn.Conv1d(1, 16, kernel_size=5, padding=2),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(16, 32, kernel_size=5, padding=2),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [62]:
device = torch.device("cuda" if torch.cuda.is_available() else "mps")

model = AttentionCNN().to(device)

print(device)

mps


In [63]:
criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [64]:
epochs = 30

for epoch in range(epochs):

    model.train()

    running_loss = 0

    correct = 0
    total = 0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device).unsqueeze(1)

        optimizer.zero_grad()

        outputs = model(X_batch)

        loss = criterion(outputs, y_batch)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        predictions = (torch.sigmoid(outputs) > 0.5).float()

        correct += (predictions == y_batch).sum().item()

        total += y_batch.size(0)

    accuracy = 100 * correct / total

    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"Loss: {running_loss/len(train_loader):.4f} | "
        f"Train Acc: {accuracy:.2f}%"
    )

Epoch 1/30 | Loss: 0.5775 | Train Acc: 69.72%
Epoch 2/30 | Loss: 0.5047 | Train Acc: 76.16%
Epoch 3/30 | Loss: 0.4816 | Train Acc: 77.02%
Epoch 4/30 | Loss: 0.4622 | Train Acc: 77.97%
Epoch 5/30 | Loss: 0.4506 | Train Acc: 79.67%
Epoch 6/30 | Loss: 0.4297 | Train Acc: 80.58%
Epoch 7/30 | Loss: 0.4299 | Train Acc: 80.51%
Epoch 8/30 | Loss: 0.4300 | Train Acc: 80.28%
Epoch 9/30 | Loss: 0.4168 | Train Acc: 81.26%
Epoch 10/30 | Loss: 0.4042 | Train Acc: 82.09%
Epoch 11/30 | Loss: 0.4105 | Train Acc: 82.28%
Epoch 12/30 | Loss: 0.3952 | Train Acc: 82.66%
Epoch 13/30 | Loss: 0.3888 | Train Acc: 82.98%
Epoch 14/30 | Loss: 0.3909 | Train Acc: 82.91%
Epoch 15/30 | Loss: 0.3842 | Train Acc: 83.45%
Epoch 16/30 | Loss: 0.3901 | Train Acc: 82.50%
Epoch 17/30 | Loss: 0.3910 | Train Acc: 82.14%
Epoch 18/30 | Loss: 0.3823 | Train Acc: 83.36%
Epoch 19/30 | Loss: 0.3831 | Train Acc: 83.09%
Epoch 20/30 | Loss: 0.3669 | Train Acc: 83.84%
Epoch 21/30 | Loss: 0.3798 | Train Acc: 82.55%
Epoch 22/30 | Loss: 0.

In [65]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

model.eval()

y_true = []
y_pred = []

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        X_batch = X_batch.to(device)

        outputs = model(X_batch)

        predictions = (torch.sigmoid(outputs) > 0.5).float()

        y_true.extend(y_batch.numpy())

        y_pred.extend(predictions.cpu().numpy().flatten())

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print(f"Accuracy : {accuracy*100:.2f}%")
print(f"Precision: {precision*100:.2f}%")
print(f"Recall   : {recall*100:.2f}%")
print(f"F1 Score : {f1*100:.2f}%")

print("\nConfusion Matrix")
print(confusion_matrix(y_true, y_pred))

Accuracy : 82.05%
Precision: 89.98%
Recall   : 76.18%
F1 Score : 82.51%

Confusion Matrix
[[438  52]
 [146 467]]


In [66]:
import torch
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

model.eval()

# Get probabilities
all_probs = []
all_labels = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)

        outputs = model(X_batch)
        probs = torch.sigmoid(outputs).cpu().numpy().flatten()

        all_probs.extend(probs)
        all_labels.extend(y_batch.numpy())

all_probs = np.array(all_probs)
all_labels = np.array(all_labels)

thresholds = [0.35, 0.40, 0.45, 0.50, 0.55]

for t in thresholds:

    preds = (all_probs >= t).astype(int)

    acc = accuracy_score(all_labels, preds)
    prec = precision_score(all_labels, preds)
    rec = recall_score(all_labels, preds)
    f1 = f1_score(all_labels, preds)

    print("="*45)
    print(f"Threshold : {t:.2f}")
    print(f"Accuracy  : {acc*100:.2f}%")
    print(f"Precision : {prec*100:.2f}%")
    print(f"Recall    : {rec*100:.2f}%")
    print(f"F1 Score  : {f1*100:.2f}%")
    print("Confusion Matrix")
    print(confusion_matrix(all_labels, preds))

Threshold : 0.35
Accuracy  : 84.04%
Precision : 86.24%
Recall    : 84.83%
F1 Score  : 85.53%
Confusion Matrix
[[407  83]
 [ 93 520]]
Threshold : 0.40
Accuracy  : 83.68%
Precision : 87.39%
Recall    : 82.54%
F1 Score  : 84.90%
Confusion Matrix
[[417  73]
 [107 506]]
Threshold : 0.45
Accuracy  : 82.86%
Precision : 88.69%
Recall    : 79.28%
F1 Score  : 83.72%
Confusion Matrix
[[428  62]
 [127 486]]
Threshold : 0.50
Accuracy  : 82.05%
Precision : 89.98%
Recall    : 76.18%
F1 Score  : 82.51%
Confusion Matrix
[[438  52]
 [146 467]]
Threshold : 0.55
Accuracy  : 81.41%
Precision : 90.96%
Recall    : 73.90%
F1 Score  : 81.55%
Confusion Matrix
[[445  45]
 [160 453]]


In [ ]:
torch.save(model.state_dict(), "attention_cnn.pth")
print("Model saved successfully!")

Model saved successfully!
